# P1-A8: Semantic Relationships and Retrieval

### Embedding Models

#### Small (~384 dimensions)
BAAI/bge-small-en-v1.5
https://huggingface.co/BAAI/bge-small-en-v1.5

#### Medium (~768 dimensions)
prdev/mini-gte
https://huggingface.co/spaces/mteb/leaderboard

#### Large (1024-1536 dimensions)
BAAI/bge-large-en
https://huggingface.co/spaces/mteb/leaderboard

#### Model from previous assignment
Qwen/Qwen2.5-1.5B-Instruct
https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct


In [107]:
# Small
import numpy as np
from transformers import pipeline

pipe = pipeline("feature-extraction", model="BAAI/bge-small-en-v1.5")
output = np.array(pipe("test sentence"))

Device set to use cpu


In [108]:
print(output.shape) # 384 dimensions

(1, 4, 384)


In [109]:
# Medium
from sentence_transformers import SentenceTransformer

med_output = SentenceTransformer("prdev/mini-gte")
print(med_output.get_sentence_embedding_dimension()) # 768 dimensions

768


In [110]:
# Large
large_output = SentenceTransformer("BAAI/bge-large-en")
print(large_output.get_sentence_embedding_dimension()) # 1024 dimensions


1024


# RAG Pipeline


#### Knowledge Base

I wasn't sure how to find a social / activites dataset so I searched the web for inputs on a wide variety of topics and social settings

In [111]:
import pandas as pd
df = pd.read_csv("knowledge_base_clean.csv")
df.sample(n=3)

,id,topic,source,title,url,text,words,notes
21,33,sports_first_time,Reddit,NaN,NaN,To not compare your abilities with others. The...,132,NaN
2,3,running_clubs,Runner's World,NaN,NaN,"Initially, The San Francisco Road Runners Club...",137,NaN
36,32,social_events,Reddit,NaN,NaN,How many bars are you looking at going too and...,99,NaN


In [112]:
# Changing the df to a more usable data structure

sections = []

for row in df.itertuples():
    sections.append((row.topic, row.text))



In [113]:
source_text='\n\n'.join(f'{title}. {body}' for title,body in sections)
print(source_text)


running_clubs. Jessamy Little, who founded the Cass Runners Club, a 100-plus person running group in London comprised of her business school classmates, suggests asking potential members what days, times, and locations work best with their schedules. Some groups may favor an early morning sweat sesh, while others may prefer meeting after work. “A recommendation for a newer club is to have two set running days,” Little says. “One during the week that is more focused on ‘getting it done’ and one on weekends that can have a more ‘fun and footloose’ vibe.” For Little’s group, the weekend runs were geared toward exploring new areas of the city.

running_clubs. If you have new people coming in, you can’t assume that they know the rules and guidelines. Communication needs to be all the time. Even though the group’s weather-proof policy may be well understood among current members, every time the skies get gloomy, the club blasts their social channels with reminders that the runs are still on.

#### Chunking

##### Fixed Chunking

In [114]:
# Template from week 8's scripts (cell 22)

# Selected 52 per chunk because the data is larger than the examples in the week 8 scripts. I wanted to pick a large number to ensure that each chunk has a complete thought.
def fixed_chunks(text, words_per_chunk=52):
    words=text.split()
    return [' '.join(words[i:i+words_per_chunk]) for i in range(0,len(words),words_per_chunk)]

# Print a few chunks and inspect where sentence boundaries get cut.
fixed=fixed_chunks(source_text)
for i,c in enumerate(fixed[:5],1): print(f'Chunk {i}:',c)
print('Warning: naive word-based fixed chunking can split sentences.')


Chunk 1: running_clubs. Jessamy Little, who founded the Cass Runners Club, a 100-plus person running group in London comprised of her business school classmates, suggests asking potential members what days, times, and locations work best with their schedules. Some groups may favor an early morning sweat sesh, while others may prefer meeting after work.
Chunk 2: “A recommendation for a newer club is to have two set running days,” Little says. “One during the week that is more focused on ‘getting it done’ and one on weekends that can have a more ‘fun and footloose’ vibe.” For Little’s group, the weekend runs were geared toward exploring new areas
Chunk 3: of the city. running_clubs. If you have new people coming in, you can’t assume that they know the rules and guidelines. Communication needs to be all the time. Even though the group’s weather-proof policy may be well understood among current members, every time the skies get gloomy, the club blasts their social
Chunk 4: channels with re

##### Overlapping Chunks

In [115]:
# Template from week 8 scripts (cell 23)

# In this segment, I increased the total to ensure that the chunks would have complete thoughts, c[:150]
paragraphs=[p.strip() for p in source_text.split('\n\n') if p.strip()]

# Create chunks containing adjacent paragraphs with overlap.
def overlap_paragraph_chunks(paragraphs, window=2, stride=1):
    return ['\n'.join(paragraphs[i:i+window]) for i in range(0,len(paragraphs),stride) if paragraphs[i:i+window]]

# Print each chunk to see repeated context across neighboring chunks.
overlap=overlap_paragraph_chunks(paragraphs,window=2,stride=1)
for i,c in enumerate(overlap,1): print(f'Chunk {i}:',c[:150].replace('\n',' / '),'...')

Chunk 1: running_clubs. Jessamy Little, who founded the Cass Runners Club, a 100-plus person running group in London comprised of her business school classmate ...
Chunk 2: running_clubs. If you have new people coming in, you can’t assume that they know the rules and guidelines. Communication needs to be all the time. Eve ...
Chunk 3: running_clubs. Initially, The San Francisco Road Runners Club, a 400-member nonprofit group founded in 2001, didn’t seek sponsorships. Instead, it bui ...
Chunk 4: making_friends. I heard that the best way to meet people is not just to be doing something at the same time but to be starting something at the same t ...
Chunk 5: making_friends. As someone who has been going to run clubs for 5 years now, and has been to at least over 15 of them now, I find that you tend to make ...
Chunk 6: making_friends. What I found to be effective was first consistently showing up to smaller/medium sized run clubs and then overtime gradually starting  ...
Chunk 7: making_

##### Hybrid - 3 sentence

In [116]:
# For the hybrid strategy, I decided to use the nltk library to chunk after the third "." in the paragraph.

# This is similar reasoning to the previous chunking strategies. I wanted to select a value high enough where there would be complete thoughts but not too many to the point of defeating the purpose of chunking.
import nltk
def chunk_three_period(text):

    sentences = nltk.tokenize.sent_tokenize(text)

    # Group sentences into pairs
    chunks = [" ".join(sentences[i:i+3]) for i in range(0, len(sentences), 3)]
    return chunks

chunks = []
for title, body in sections:
    for chunk in chunk_three_period(body):
        chunks.append({"topic": title, "text": chunk})

In [117]:
chunks[1]

{'topic': 'running_clubs',
 'text': '“One during the week that is more focused on ‘getting it done’ and one on weekends that can have a more ‘fun and footloose’ vibe.” For Little’s group, the weekend runs were geared toward exploring new areas of the city.'}

### Embedding Pipeline

•Small: https://huggingface.co/BAAI/bge-small-en-v1.5

•Medium: https://huggingface.co/spaces/mteb/leaderboard

•Large: https://huggingface.co/spaces/mteb/leaderboard


##### Small model

In [118]:
import faiss
import numpy as np

def build_index(embeddings):
    emb = np.asarray(embeddings, dtype="float32")
    faiss.normalize_L2(emb)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    return index

small_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
embeddings_fix = small_model.encode(fixed)
embeddings_overlap = small_model.encode(overlap)
embedding_hybrid = small_model.encode(chunks)


index_small = build_index(embeddings_fix)
index_small_overlap = build_index(embeddings_overlap)
index_small_hybrid = build_index(embedding_hybrid)


##### Medium model

In [119]:
med_model = SentenceTransformer("prdev/mini-gte")
embeddings_fix_med = med_model.encode(fixed)
embeddings_overlap_med = med_model.encode(overlap)
embeddings_hybrid_med = med_model.encode(chunks)


index_med_fix = build_index(embeddings_fix_med)
index_med_overlap = build_index(embeddings_overlap_med)
index_med_hybrid = build_index(embeddings_hybrid_med)


##### Large model

In [120]:
model = SentenceTransformer("BAAI/bge-large-en")
embedding_fix_large = model.encode(fixed)
embedding_overlap_large = model.encode(overlap)
embedding_hybrid_large = model.encode(chunks)


index_large_fix = build_index(embedding_fix_large)
index_large_overlap = build_index(embedding_overlap_large)
index_large_hybrid = build_index(embedding_hybrid_large)


### Retrieval System

In [123]:
instruct = "Search for the closest aligning paragraph: "

def retrieve(query, model, index, chunk_texts, k=3, prefix=instruct):
    qvec = model.encode([prefix + query], normalize_embeddings=True)
    scores, ids = index.search(np.asarray(qvec, dtype="float32"), k)
    # sim

    return [(chunk_texts[i], float(s)) for i, s in zip(ids[0], scores[0])]
    # K near

### Generation Pipeline

In [126]:
from transformers import pipeline

a6_model = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct")

query = "How should I prepare for my first hike?"
# query

retrieved = retrieve(query, model, index_large_fix, fixed, 3)

system = "Answer using only the provided text."
# instructions

context = "\n\n".join(text for text, _ in retrieved)

messages = [
    {"role": "system", "content": system},
    {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
]

result = a6_model(messages, max_new_tokens=200, do_sample=False)
print(result[0]["generated_text"][-1]["content"])


Device set to use cpu


Start by bringing a pocket knife, an emergency kit containing items like bandaids, gauze, antihistamines, anti-inflammatory medications, ointments for bug bites, small scissors, and other essentials, especially if you're planning to go into remote areas. Additionally, consider buying a simple tent since it's easier to set up than a full-sized tent. For your lunch, opt for sandwiches from the cooler or skip the cooler altogether and enjoy peanut butter. Lastly, don't rush; take things slow and gradually build up your skills and experience.


# Part 2: System Experiments

##### Test Queries

In [ ]:
tests=[
    {'query':'Advice for going on my first hike','relevant':{24,9, 25}},
    {'query':'What should we cook tonight','relevant':{28,29, 18}},
    {'query':'How to get into card collecting','relevant':{21,22}},
    {'query':'How to get a running group','relevant':{1,2,3}},
    {'query':'First time rock climbing advice','relevant':{33}},
     {'query': 'How do I put myself out there socially?','relevant':{4,5,7,8}},
    {'query': "What is a good beginner Catan strategy?", 'relevant': {23}},
]

In [127]:
df

,id,topic,source,title,url,text,words,notes
0,1,running_clubs,Runner's World,NaN,NaN,"Jessamy Little, who founded the Cass Runners C...",106,NaN
1,2,running_clubs,Runner's World,NaN,NaN,"If you have new people coming in, you can’t as...",81,fixed stray quote marks
2,3,running_clubs,Runner's World,NaN,NaN,"Initially, The San Francisco Road Runners Club...",137,NaN
3,4,making_friends,Reddit,NaN,NaN,I heard that the best way to meet people is no...,91,NaN
4,5,making_friends,Reddit,NaN,NaN,As someone who has been going to run clubs for...,191,NaN
5,7,making_friends,Reddit,NaN,NaN,What I found to be effective was first consist...,119,NaN
6,8,making_friends,Reddit,NaN,NaN,I am a member of one of the bigger run clubs a...,124,NaN
7,9,making_friends,Reddit,NaN,NaN,Mindset is the first issue that jumps out at m...,109,NaN
8,10,making_friends,Zenhabits,NaN,NaN,So practice socializing with friends or family...,125,NaN
9,6,hiking_camping,Reddit,NaN,NaN,Besides the more obvious stuff like trail info...,105,NaN
